# Ambiente de Testes SQL com DuckDB

Este notebook é uma plataforma de laboratório para testar consultas SQL com DuckDB de forma ampla e profissional.

- Carrega automaticamente todos os arquivos CSV na pasta `data/`.
- Registra cada arquivo como uma tabela DuckDB para criar um ambiente relacional completo.
- Permite criar e validar consultas de seleção, agregação, joins, funções de janela e análises exploratórias.

> Além disso, existe uma pasta `sql_commands/` no repositório para que você salve seus scripts e exemplos SQL em arquivos `.sql` ou `.txt`. Isso torna seu material de aula mais organizado e reutilizável.


## 1. Preparar o ambiente

Execute esta célula para carregar as bibliotecas necessárias e criar uma função de apoio para executar consultas SQL no DuckDB.

Pressione Ctrl + Shift + P e digite "Jupyter: Select Kernel"
Escolha "Python Local"
Clique em "Restart Kernel" ou recarregue o arquivo

In [2]:
import importlib.util
import subprocess
import sys

#for pkg in ('duckdb', 'pandas'):
#    if importlib.util.find_spec(pkg) is None:
#        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])



for pkg in ('duckdb', 'pandas', 'ipykernel'):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])


import duckdb
import pandas as pd
from pathlib import Path

# Diretório de dados local
data_dir = Path('../data')
if not data_dir.exists():
    raise FileNotFoundError(f'Diretório não encontrado: {data_dir}')

csv_files = sorted(data_dir.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'Nenhum arquivo CSV encontrado em: {data_dir}')

# Conexão local em memória
con = duckdb.connect(database=':memory:')
loaded_tables = []
for csv_path in csv_files:
    table_name = csv_path.stem.lower()
    df = pd.read_csv(csv_path)
    con.register(table_name, df)
    loaded_tables.append(table_name)

last_result = None

def sql(query: str):
    global last_result
    last_result = con.execute(query).df()
    display(last_result)

## 2. Carregar os dados locais

Este notebook carrega todos os arquivos CSV presentes em `data/`.
Cada arquivo é registrado automaticamente no DuckDB como uma tabela cujo nome é o nome do arquivo (sem extensão).

Dessa forma, você pode testar `JOINs` e outras consultas SQL entre tabelas diferentes.

Por exemplo:

- `cliente.csv` → tabela `cliente`
- `produto.csv` → tabela `produto`
- `vendedor.csv` → tabela `vendedor`
- `vendas.csv` → tabela `vendas`

In [3]:
# Exibe as tabelas carregadas e um exemplo de dados - {loaded_tables[n]}
print(f'Arquivos CSV carregados como tabelas: {", ".join(loaded_tables)}')
print(f'Tabela de exemplo: {loaded_tables[1]}')
display(con.execute(f"SELECT * FROM {loaded_tables[1]} LIMIT 5").df())


Arquivos CSV carregados como tabelas: baby_names, menu_items, order_details, restaurant_db_data_dictionary, student_grades, students
Tabela de exemplo: menu_items


,menu_item_id,item_name,category,price
0,101,Hamburger,American,12.95
1,102,Cheeseburger,American,13.95
2,103,Hot Dog,American,9.00
3,104,Veggie Burger,American,10.50
4,105,Mac & Cheese,American,7.00


### Duplique este notebook para começar outro exercício e insira suas consultas SQL abaixo: (divirta-se :-) .

### 💡 Modelos de Referência (Snippets)

Utilize os padrões abaixo para construir suas análises:

**1. Filtro e Ordenação:**
```sql
sql("""
SELECT Nome, Total
FROM baby_names
WHERE Sexo = 'Menina'
ORDER BY Total DESC
""" )
```

**2. Agregação e Sumarização:**
```sql
sql("""
SELECT Sexo, COUNT(*) as Qtd_Nomes, SUM(Total) as Total_Bebes
FROM baby_names
GROUP BY Sexo
""" )
```

---

 Proj : Basic SQL 

In [4]:
# 1 & 2. view the table (SELECT, FROM )
sql('''
    SELECT *
FROM students;
    ''')

,id,student_name,grade_level,gpa,school_lunch,birthday,email
0,1,Abby Johnson,10,3.1,Yes,5/14/08,abby.johnson@mavenhighschool.com
1,2,Bob Smith,11,3.1,No,9/30/07,bob.smith@mavenhighschool.com
2,3,Catherine Davis,12,3.6,Yes,11/21/06,catherine.davis@mavenhighschool.com
3,4,Daniel Brown,9,3.5,Yes,3/10/09,daniel.brown@mavenhighschool.com
4,5,Eva Martinez,10,2.7,No,2/5/08,eva.martinez@mavenhighschool.com
5,6,Frank Wilson,11,3.2,No,7/17/07,frank.wilson@mavenhighschool.com
6,7,Grace Lee,12,3.0,Yes,12/2/06,grace.lee@mavenhighschool.com
7,8,Henry Taylor,9,3.0,Yes,6/8/09,henry.taylor@mavenhighschool.com
8,9,Isabella Moore,10,2.8,Yes,1/19/08,isabella.moore@mavenhighschool.com
9,10,Jack Thompson,11,2.9,Yes,4/25/07,jack.thompson@mavenhighschool.com


In [5]:
# 1 & . View the table (SELECT, FROM )
sql('''
    SElECT student_name, gpa, school_lunch
    FROM  students;
    ''')

,student_name,gpa,school_lunch
0,Abby Johnson,3.1,Yes
1,Bob Smith,3.1,No
2,Catherine Davis,3.6,Yes
3,Daniel Brown,3.5,Yes
4,Eva Martinez,2.7,No
5,Frank Wilson,3.2,No
6,Grace Lee,3.0,Yes
7,Henry Taylor,3.0,Yes
8,Isabella Moore,2.8,Yes
9,Jack Thompson,2.9,Yes


In [8]:
# 3. show stundents who get scool lunch (WHERE)
sql('''
    SELECT student_name, gpa, school_lunch
    FROM students
    WHERE school_lunch = 'Yes' AND gpa > 3.3;
    ''')

,student_name,gpa,school_lunch
0,Catherine Davis,3.6,Yes
1,Daniel Brown,3.5,Yes
2,Liam Green,4.0,Yes
3,Olivia Adams,3.7,Yes


In [12]:
# 4. short the students by gpa (ORDER BY)
sql('''
    SELECT student_name, gpa, school_lunch
    FROM students
    WHERE school_lunch = 'Yes' AND gpa > 3.3
    ORDER BY gpa DESC;
    ''')

,student_name,gpa,school_lunch
0,Liam Green,4.0,Yes
1,Olivia Adams,3.7,Yes
2,Catherine Davis,3.6,Yes
3,Daniel Brown,3.5,Yes


In [31]:
# 5.1 show the average gpa for each grade level (GROUP BY )
sql('''
    SELECT grade_level, AVG(gpa) AS avg_gpa
    FROM students
    GROUP BY grade_level
    ORDER BY avg_gpa DESC;
    ''')

,grade_level,avg_gpa
0,9,3.400000
1,12,3.166667
2,10,3.150000
3,11,3.050000


In [32]:
# 5.2 show the average gpa for each grade level MAX
sql('''
    SELECT grade_level, AVG(gpa) AS avg_gpa, MAX(gpa) AS max_gpa, MIN(gpa) AS min_gpa
    FROM students
    GROUP BY grade_level
    ORDER BY avg_gpa DESC;
    ''')

,grade_level,avg_gpa,max_gpa,min_gpa
0,9,3.400000,3.7,3.0
1,12,3.166667,3.6,2.9
2,10,3.150000,4.0,2.7
3,11,3.050000,3.2,2.9


In [34]:
# 6 show the grade levels with an averange gpa below 3.3 (HAVING)
sql('''
    SELECT grade_level, AVG(gpa) AS avg_gpa
    FROM students
    GROUP BY grade_level
    HAVING avg_gpa < 3.3
    ORDER BY avg_gpa DESC;
    ''')

,grade_level,avg_gpa
0,12,3.166667
1,10,3.150000
2,11,3.050000


In [38]:
# 7. special keywords : LIMIT | COUNT | DISTINCT
sql('''
    SELECT COUNT (*) -- student_name, gpa, school_lunch
    FROM students
    WHERE school_lunch = 'Yes' AND gpa > 3.3;
    ''')    

,count_star()
0,4


In [50]:
# 7.1 special keywords : LIMIT | COUNT | DISTINCT
sql('''
    SELECT  DISTINCT gpa
    FROM students
    ORDER BY gpa DESC;
    ''')    

,gpa
0,4.0
1,3.7
2,3.6
3,3.5
4,3.4
5,3.2
6,3.1
7,3.0
8,2.9
9,2.8


In [52]:
# 8. show the final grades for each student: LEFT JOIN
sql('''
    SELECT *
    FROM students;
''')

sql('''
    SELECT *
    FROM student_grades;
''')

,id,student_name,grade_level,gpa,school_lunch,birthday,email
0,1,Abby Johnson,10,3.1,Yes,5/14/08,abby.johnson@mavenhighschool.com
1,2,Bob Smith,11,3.1,No,9/30/07,bob.smith@mavenhighschool.com
2,3,Catherine Davis,12,3.6,Yes,11/21/06,catherine.davis@mavenhighschool.com
3,4,Daniel Brown,9,3.5,Yes,3/10/09,daniel.brown@mavenhighschool.com
4,5,Eva Martinez,10,2.7,No,2/5/08,eva.martinez@mavenhighschool.com
5,6,Frank Wilson,11,3.2,No,7/17/07,frank.wilson@mavenhighschool.com
6,7,Grace Lee,12,3.0,Yes,12/2/06,grace.lee@mavenhighschool.com
7,8,Henry Taylor,9,3.0,Yes,6/8/09,henry.taylor@mavenhighschool.com
8,9,Isabella Moore,10,2.8,Yes,1/19/08,isabella.moore@mavenhighschool.com
9,10,Jack Thompson,11,2.9,Yes,4/25/07,jack.thompson@mavenhighschool.com


,semester_id,class_id,department,class_name,student_id,final_grade
0,SPR_2024,101,Math,Algebra,4,85.0
1,SPR_2024,101,Math,Algebra,8,76.0
2,SPR_2024,101,Math,Algebra,11,90.0
3,SPR_2024,101,Math,Algebra,15,97.0
4,SPR_2024,102,Math,Geometry,1,93.0
...,...,...,...,...,...,...
57,SPR_2024,404,General,Senior Seminar,3,100.0
58,SPR_2024,404,General,Senior Seminar,7,98.0
59,SPR_2024,404,General,Senior Seminar,16,95.0
60,SPR_2024,404,General,Senior Seminar,17,NaN


In [56]:
# 8.1 show the final grades for each student: LEFT JOIN
sql('''
    SELECT *
    FROM students  LEFT JOIN student_grades
        ON students.id = student_grades.student_id;
''')

,id,student_name,grade_level,gpa,school_lunch,birthday,email,semester_id,class_id,department,class_name,student_id,final_grade
0,4,Daniel Brown,9,3.5,Yes,3/10/09,daniel.brown@mavenhighschool.com,SPR_2024,101,Math,Algebra,4,85.0
1,8,Henry Taylor,9,3.0,Yes,6/8/09,henry.taylor@mavenhighschool.com,SPR_2024,101,Math,Algebra,8,76.0
2,11,Karen White,9,3.4,No,9/10/09,karen.white@mavenhighschool.com,SPR_2024,101,Math,Algebra,11,90.0
3,15,Olivia Adams,9,3.7,Yes,12/11/09,olivia.adams@mavenhighschool.com,SPR_2024,101,Math,Algebra,15,97.0
4,1,Abby Johnson,10,3.1,Yes,5/14/08,abby.johnson@mavenhighschool.com,SPR_2024,102,Math,Geometry,1,93.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,15,Olivia Adams,9,3.7,Yes,12/11/09,olivia.adams@mavenhighschool.com,SPR_2024,401,General,Physical Education,15,95.0
57,3,Catherine Davis,12,3.6,Yes,11/21/06,catherine.davis@mavenhighschool.com,SPR_2024,404,General,Senior Seminar,3,100.0
58,7,Grace Lee,12,3.0,Yes,12/2/06,grace.lee@mavenhighschool.com,SPR_2024,404,General,Senior Seminar,7,98.0
59,16,Peter Park,12,2.9,Yes,2/11/06,peter.park@mavenhighschool.com,SPR_2024,404,General,Senior Seminar,16,95.0


In [ ]:
# 8.1 show the final grades for each student: LEFT JOIN + AS 
sql('''
    SELECT *
    FROM students AS s LEFT JOIN student_grades AS sg
        ON s.id = sg.student_id;
''')

,id,student_name,grade_level,gpa,school_lunch,birthday,email,semester_id,class_id,department,class_name,student_id,final_grade
0,4,Daniel Brown,9,3.5,Yes,3/10/09,daniel.brown@mavenhighschool.com,SPR_2024,101,Math,Algebra,4,85.0
1,8,Henry Taylor,9,3.0,Yes,6/8/09,henry.taylor@mavenhighschool.com,SPR_2024,101,Math,Algebra,8,76.0
2,11,Karen White,9,3.4,No,9/10/09,karen.white@mavenhighschool.com,SPR_2024,101,Math,Algebra,11,90.0
3,15,Olivia Adams,9,3.7,Yes,12/11/09,olivia.adams@mavenhighschool.com,SPR_2024,101,Math,Algebra,15,97.0
4,1,Abby Johnson,10,3.1,Yes,5/14/08,abby.johnson@mavenhighschool.com,SPR_2024,102,Math,Geometry,1,93.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,15,Olivia Adams,9,3.7,Yes,12/11/09,olivia.adams@mavenhighschool.com,SPR_2024,401,General,Physical Education,15,95.0
57,3,Catherine Davis,12,3.6,Yes,11/21/06,catherine.davis@mavenhighschool.com,SPR_2024,404,General,Senior Seminar,3,100.0
58,7,Grace Lee,12,3.0,Yes,12/2/06,grace.lee@mavenhighschool.com,SPR_2024,404,General,Senior Seminar,7,98.0
59,16,Peter Park,12,2.9,Yes,2/11/06,peter.park@mavenhighschool.com,SPR_2024,404,General,Senior Seminar,16,95.0


In [60]:
# 8.2 show the final grades for each student: LEFT JOIN + AS 
sql('''
    SELECT s.id, s.student_name,
           sg.class_name, sg.final_grade
    FROM students AS s LEFT JOIN student_grades AS sg
        ON s.id = sg.student_id;
''')

,id,student_name,class_name,final_grade
0,4,Daniel Brown,Algebra,85.0
1,8,Henry Taylor,Algebra,76.0
2,11,Karen White,Algebra,90.0
3,15,Olivia Adams,Algebra,97.0
4,1,Abby Johnson,Geometry,93.0
...,...,...,...,...
56,15,Olivia Adams,Physical Education,95.0
57,3,Catherine Davis,Senior Seminar,100.0
58,7,Grace Lee,Senior Seminar,98.0
59,16,Peter Park,Senior Seminar,95.0
